<a href="https://colab.research.google.com/github/BenMillerDev/Applied-LLM-Systems/blob/week-8-capstone/week-8/week8_capstone.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 8 Assignment | Integrated LLM System: NailSalonApp Owner Assistant
## COSC 650 Applied LLM Systems
### Ben Miller

# Part 1: Integrate
Combine at least three techniques from earlier weeks (prompt evaluation, retrieval-augmented generation, tool use, or dynamic few-shot) into a single system that solves a real problem. Reuse your prior components rather than rebuilding them, and cite in the README which week each came from.

### Load the Gemini API key from Colab secrets

Reused from Week 4 as is.

In [ ]:
import os

try:
    from google.colab import userdata
    _key = userdata.get('GEMINI_API_KEY')
    if _key:
        os.environ['GEMINI_API_KEY'] = _key
except Exception:
    pass  # not running in Colab, or the secret isn't set there -- fall back to whatever's already in the environment

HAS_API_KEY = bool(os.environ.get('GEMINI_API_KEY', '').strip())
print('GEMINI_API_KEY set:', HAS_API_KEY)

### Constants used across the notebook

In [ ]:
import ast, json, math, operator, pathlib, re, shutil, signal, textwrap
from datetime import datetime, timedelta

# Models
MODEL = 'gemini-3.5-flash-lite'   # the assistant itself, plus HyDE
JUDGE_MODEL = 'gemini-3.5-flash'  # a larger model grades the answers, so the assistant isn't grading its own work
TEMPERATURE = 0.0                 # same question -> same answer, so eval scores don't drift between runs
EMBED_MODEL_NAME = 'sentence-transformers/all-MiniLM-L6-v2'  # Week 5
RERANK_MODEL_NAME = 'cross-encoder/ms-marco-MiniLM-L-6-v2'   # Week 6

# Retrieval (Weeks 5 and 6)
TOP_K = 3         # chunks handed to the model per search_docs call
CANDIDATE_N = 12  # chunks FAISS pulls before the cross-encoder reranks them down to TOP_K
USE_HYDE = True   # starting values; Part 2's retrieval comparison decides the final settings
USE_RERANK = True

# Prompt (Week 3)
PROMPT_VERSION = 'v1'  # which prompts/assistant_<version>.md file the assistant runs with

# Agent loop (Week 4)
MAX_TURNS = 6

# Tool safety
# In the real app this would be the uid from the owner's verified Firebase Auth
# token. It's passed to the tools by dispatch(), never by the model, so a prompt
# can't point a tool at another salon's data.
OWNER_ID = 'salon-owner-1'
ALLOW_WRITE_TOOLS = False  # read-only for now: dispatch() refuses any tool not marked 'access': 'read'

# timeSlots.ts reads the clock with `new Date()`. Here "now" is fixed so the same
# question gets the same open slots on every run. 2026-10-12 is a Monday.
NOW = datetime(2026, 10, 12, 10, 0)

# Booking rules hard-coded in timeSlots.ts (not owner settings)
SLOT_INCREMENT_MINUTES = 15   # candidate start times step every 15 minutes
DEFAULT_BUFFER_MINUTES = 15   # bufferTime's default when the owner hasn't set one
MINIMUM_ADVANCE_MINUTES = 60  # same-day bookings must start more than 1 hour from now
BOOKED_STATUSES = ['pending', 'confirmed']  # appointments that still hold their time (cancelled and completed don't)

# Paths
REPO_URL = 'https://github.com/BenMillerDev/Applied-LLM-Systems.git'
BRANCH = 'week-8-capstone'
DOCS_DIR = pathlib.Path('docs')
PROMPTS_DIR = pathlib.Path('prompts')

### Download documents and prompts from my repo

The corpus is the same 11 NailSalonApp documents from Week 5, one per subsystem of the app. The assistant's system prompt is a versioned file in this week's `prompts/` folder, the same way Week 3 kept its prompts.

In [ ]:
if not DOCS_DIR.exists() or not PROMPTS_DIR.exists():
    # docs/ and prompts/ aren't sitting next to this notebook already (e.g. it
    # was opened on its own rather than as part of a full repo clone) -- pull them down
    !rm -rf _repo
    !git clone -b {BRANCH} {REPO_URL} _repo
    !cp -r _repo/week-5/docs .
    !cp -r _repo/week-8/prompts .

DOCS = {doc_path.stem: doc_path.read_text() for doc_path in sorted(DOCS_DIR.glob('*.md'))}

print(f'loaded {len(DOCS)} docs:', list(DOCS.keys()))
print(f'total chars: {sum(len(doc_text) for doc_text in DOCS.values())}')

### Save model responses so reruns give the same results

Every model response (HyDE passages now, full assistant runs later) is saved in Google Drive to JSON the first time it's made and reused after that. Reruns don't spend API quota, and Part 2's scores can't drift between runs. The saved runs from my own run are committed in the repo's `week-8/cache/` folder, so someone without an API key still sees real results.

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    CACHE_DIR = pathlib.Path('/content/drive/MyDrive/cosc650_week8/cache')
except ImportError:
    CACHE_DIR = pathlib.Path('cache')  # not in Colab: keep it next to the notebook
CACHE_DIR.mkdir(parents=True, exist_ok=True)

# seed the cache with the committed runs from the repo, without overwriting newer ones
REPO_CACHE_DIR = pathlib.Path('_repo/week-8/cache')
if REPO_CACHE_DIR.exists():
    for repo_cache_file in REPO_CACHE_DIR.glob('*.json'):
        local_copy = CACHE_DIR / repo_cache_file.name
        if not local_copy.exists():
            shutil.copy(repo_cache_file, local_copy)


def load_cache(cache_name):
    """Every saved response of one kind ('hyde', 'runs', ...), as a dict keyed by a string describing the call."""
    cache_path = CACHE_DIR / f'{cache_name}.json'
    if not cache_path.exists():
        return {}
    return json.loads(cache_path.read_text())


def save_cache(cache_name, cache):
    """Write one kind of saved response back to disk."""
    cache_path = CACHE_DIR / f'{cache_name}.json'
    cache_path.write_text(json.dumps(cache, indent=2))


print('cache folder:', CACHE_DIR)
print('saved caches:', sorted(cache_file.name for cache_file in CACHE_DIR.glob('*.json')))

### Gemini helper for single prompts

Reused from Weeks 5 and 6: Gemini through its OpenAI-compatible endpoint, returning `None` when there's no API key. HyDE uses this. The assistant itself uses `google-genai` instead, because that's the client Week 4's tool-calling loop was built on.

In [ ]:
!pip install -q openai

def gemini_chat(messages, model=MODEL, **kw):
    """Gemini via the OpenAI-compatible endpoint. Returns text, or None if no key."""
    key = os.environ.get('GEMINI_API_KEY')
    if not key:
        return None
    from openai import OpenAI
    client = OpenAI(api_key=key, base_url='https://generativelanguage.googleapis.com/v1beta/openai/')
    return client.chat.completions.create(model=model, messages=messages, **kw).choices[0].message.content

## Retrieval: the `search_docs` tool (Weeks 5 and 6)

### Set up the embedder and cross-encoder

In [ ]:
!pip install -q faiss-cpu sentence-transformers

# HF_HOME has to be set before sentence_transformers is imported, or the models download to the default cache instead
os.environ['HF_HOME'] = str((pathlib.Path('.') / '.hf_cache').resolve())

import faiss
import numpy as np
from sentence_transformers import SentenceTransformer, CrossEncoder

embedder = SentenceTransformer(EMBED_MODEL_NAME)
reranker = CrossEncoder(RERANK_MODEL_NAME)

### Chunk the docs by paragraph and build the index

Paragraph chunking was Week 5's best configuration (recall@3 of 1.00), so it's the only one used here. `chunk_paragraph` is from Week 5; `build_chunks` and `build_index` are Week 6's versions of the same functions.

In [ ]:
def chunk_paragraph(text):
    """Split text into one chunk per paragraph (blank-line-separated)."""
    raw_paragraphs = text.split('\n\n')
    paragraphs = []
    for paragraph in raw_paragraphs:
        cleaned_paragraph = re.sub(r'\s+', ' ', paragraph).strip() # Collapse newlines and repeated spaces into single spaces
        if cleaned_paragraph:  # skip empty paragraphs, e.g. a leading or trailing blank line
            paragraphs.append(cleaned_paragraph)
    return paragraphs


def build_chunks(docs, chunk_fn):
    """
    Chunk each document separately (not one concatenated blob) so a chunk
    never straddles two different docs, and tag each chunk with the doc
    name it came from -- search_docs returns that name so the assistant can
    cite it, and the relevance labeling below needs it too.
    """
    all_chunks = []
    chunk_sources = []
    for doc_name, doc_text in docs.items():
        doc_chunks = chunk_fn(doc_text)
        all_chunks.extend(doc_chunks)
        chunk_sources.extend([doc_name] * len(doc_chunks))
    return all_chunks, chunk_sources


def build_index(chunks):
    """Embed a list of chunks and load them into a FAISS index for similarity search."""
    chunk_embeddings = embedder.encode(chunks, normalize_embeddings=True).astype('float32')
    index = faiss.IndexFlatIP(chunk_embeddings.shape[1])
    index.add(chunk_embeddings)
    return index


chunks, chunk_sources = build_chunks(DOCS, chunk_paragraph)
index = build_index(chunks)
print(f"by-paragraph chunking (Week 5's best config) -> {len(chunks)} chunks from {len(DOCS)} docs")

### HyDE, reranking, and the `search_docs` tool

`retrieve_indices`, `hyde_generate` and `rerank` are Week 6's, with HyDE passages now saved to the cache. `retrieve_positions` replaces Week 6's `transformed_retrieve` so each technique can be switched on or off by itself: Part 3 needs that to blame the right one.

In [ ]:
def retrieve_indices(query, k):
    """Return the positions of the top-k chunks most similar to the query."""
    query_embedding = embedder.encode([query], normalize_embeddings=True).astype('float32')
    _, retrieved_positions = index.search(query_embedding, k)
    # FAISS pads with -1 when fewer than k chunks exist in the index -- drop those.
    return [position for position in retrieved_positions[0] if position >= 0]


def hyde_generate(query):
    """
    Ask an LLM to write a short hypothetical passage, in the same
    documentation style as the corpus, that would answer the query -- then
    that passage (not the literal question) is what gets embedded and
    searched. Passages are saved, so the same query always searches the same way.
    """
    hyde_cache = load_cache('hyde')
    cache_key = f'{MODEL}|{query}'
    if cache_key in hyde_cache:
        return hyde_cache[cache_key]

    prompt = (
        'Write a short passage (2-3 sentences) from internal technical '
        'documentation for a nail salon booking app that would answer this '
        'question. Write it as a factual statement, not a question, in a '
        'confident, documentation-like style -- your best guess at the '
        'specific detail, not a hedge.\n\n'
        f'Question: {query}\nPassage:'
    )
    passage = gemini_chat([{'role': 'user', 'content': prompt}], temperature=TEMPERATURE)
    if passage is None:
        return query  # no API key and nothing saved: search with the raw query, as Week 6 did

    hyde_cache[cache_key] = passage
    save_cache('hyde', hyde_cache)
    return passage


def get_score(position_and_score):
    """Sort key for (position, score) pairs."""
    return position_and_score[1]


def rerank(query, candidate_positions, k):
    """
    Score each candidate chunk against the original query with a
    cross-encoder, and return the top-k candidate positions by that score.
    """
    query_chunk_pairs = [(query, chunks[position]) for position in candidate_positions]
    scores = reranker.predict(query_chunk_pairs)
    scored_positions = sorted(zip(candidate_positions, scores), key=get_score, reverse=True)
    best_scored_positions = scored_positions[:k]
    return [position for position, score in best_scored_positions]


def retrieve_positions(query, use_hyde, use_rerank):
    """
    Positions of the TOP_K chunks for this query, using whichever Week 6
    techniques are switched on. With reranking, FAISS pulls a wider pool of
    CANDIDATE_N first, and the cross-encoder scores that pool against the
    ORIGINAL query (not the HyDE passage).
    """
    search_text = query
    if use_hyde:
        search_text = hyde_generate(query)

    if not use_rerank:
        return retrieve_indices(search_text, TOP_K)

    candidate_positions = retrieve_indices(search_text, CANDIDATE_N)
    return rerank(query, candidate_positions, TOP_K)


def search_docs(query):
    """The search_docs tool: the TOP_K passages for the model's query, each tagged with the doc it came from."""
    # USE_HYDE and USE_RERANK are read here, at call time, so flipping them for an ablation takes effect immediately
    positions = retrieve_positions(query, use_hyde=USE_HYDE, use_rerank=USE_RERANK)
    passages = []
    for position in positions:
        passages.append({'doc': chunk_sources[position], 'text': chunks[position]})
    return {'query': query, 'passages': passages}

### Check retrieval on the Week 5 questions

A quick check that paragraph chunking still finds the right passage under each retrieval setting, using Week 6's `label_relevant` and `precision_recall` and the same 11 questions from Weeks 5 and 6. These questions already shaped the Week 5 and 6 choices, so this is a sanity check, not the evaluation; Part 2 uses new held-out questions. Without an API key, HyDE falls back to the raw query, so the last column matches the rerank-only one.

In [ ]:
import pandas as pd

def label_relevant(source_doc, fact):
    """
    Find every chunk that counts as ground-truth relevant for one query: it
    has to come from the right document AND actually contain the fact's
    wording.
    """
    fact_lower = fact.lower()
    relevant_positions = []
    for position, (chunk_text, chunk_source) in enumerate(zip(chunks, chunk_sources)):
        if chunk_source == source_doc and fact_lower in chunk_text.lower():
            relevant_positions.append(position)
    return relevant_positions


def precision_recall(retrieved_positions, source_doc, fact):
    """
    Precision is how many of the retrieved chunks were actually relevant;
    recall is how many of the relevant chunks that exist were actually
    retrieved.
    """
    relevant_positions = label_relevant(source_doc, fact)
    hits = [position for position in retrieved_positions if position in relevant_positions]
    precision = len(hits) / len(retrieved_positions) if retrieved_positions else 0.0
    recall = len(hits) / len(relevant_positions) if relevant_positions else 0.0
    return precision, recall


# Query, the fact that answers it, and which doc that fact lives in -- identical to Weeks 5 and 6
WEEK5_QUESTIONS = [
    ('How far in advance must a customer book?', '1 hour', 'time_slot_algorithm'),
    ('What does the owner app use for navigation?', 'Expo Router', 'architecture'),
    ('How do new bookings appear on the dashboard without a refresh?', 'no pull-to-refresh or manual reload needed', 'realtime_sync'),
    ("Where do new owners' service lists start from?", 'default service catalog', 'service_management'),
    ('How many steps are in the customer booking flow?', 'four steps', 'customer_booking_flow'),
    ('What happens to Google sign-in when running in Expo Go?', 'mock version', 'auth'),
    ('What three time ranges can the analytics dashboard filter by?', 'This Week, This Month, and This Year', 'analytics'),
    ('What two fields track onboarding progress?', "dismissed the checklist", 'onboarding'),
    ("How is a client's fill due date determined?", 'computed fill due date', 'client_profiles'),
    ('What limits how many bookings an owner can take in a day?', 'maximum number of bookings allowed per day', 'availability'),
    ('Does the app support SMS booking confirmations today?', 'SMS booking confirmations', 'roadmap'),
]

RETRIEVAL_SETTINGS = {
    'paragraph only': {'use_hyde': False, 'use_rerank': False},
    '+ rerank':       {'use_hyde': False, 'use_rerank': True},
    '+ HyDE + rerank': {'use_hyde': True, 'use_rerank': True},
}


def build_recall_table(questions, retrieval_settings):
    """One row per question, with recall@TOP_K under each retrieval setting side by side."""
    table_rows = []
    for query, fact, source_doc in questions:
        row = {'query': query, 'source doc': source_doc}
        for setting_name, setting in retrieval_settings.items():
            positions = retrieve_positions(query, **setting)
            _, recall = precision_recall(positions, source_doc, fact)
            row[setting_name] = recall
        table_rows.append(row)
    return pd.DataFrame(table_rows)


recall_table = build_recall_table(WEEK5_QUESTIONS, RETRIEVAL_SETTINGS)
print(f'mean recall@{TOP_K}:')
for setting_name in RETRIEVAL_SETTINGS:
    print(f'  {setting_name:16s} {recall_table[setting_name].mean():.2f}')
recall_table

## Tools (Week 4)

### Mock owner data

The real app reads these from Firestore, which this notebook can't reach, so they're mocked here the same way `FAKE_REPO` stood in for a real pull request in Week 4. The shapes match the app: one availability document per owner (`availability/{ownerId}`, fields from `OwnerAvailability`), and an `appointments` collection where every appointment carries its `ownerId`.

There are two salons so the owner scoping can be tested: a tool asked about `salon-owner-1` must never see `salon-owner-2`'s hours or appointments. The appointments also hold client names and phone numbers, which no tool returns. The service and add-on durations are the app's default catalog from [`src/constants/services.ts`](https://github.com/BenMillerDev/NailSalonApp/blob/4df756e2504da6ad1782388ca573c330d029238b/src/constants/services.ts), which every new owner starts from.

In [ ]:
SERVICE_DURATIONS = {
    'classic-manicure': 45, 'gel-manicure': 60, 'acrylic-full-set': 90,
    'acrylic-fill': 60, 'classic-pedicure': 60, 'gel-pedicure': 75,
    'gel-removal': 30, 'acrylic-removal': 45,
    'lash-classic-full': 120, 'lash-hybrid-full': 135, 'lash-volume-full': 150,
    'lash-classic-fill': 60, 'lash-hybrid-fill': 75, 'lash-volume-fill': 90,
    'lash-removal': 30,
    'brow-wax': 20, 'brow-lamination': 45, 'brow-tint': 20,
}
ADD_ON_DURATIONS = {
    'nail-art': 15, 'ombre': 15, 'chrome-powder': 10, 'paraffin-wax': 15,
    'callus-removal': 15, 'shape-change': 10, 'gel-removal': 30,
    'lash-bath': 10, 'brow-tint': 20, 'brow-wax': 20,
}
NAIL_LENGTH_EXTRA_MINUTES = {'short': 0, 'medium': 0, 'long': 15, 'xl': 20}

# availability/{ownerId}
MOCK_AVAILABILITY_BY_OWNER = {
    'salon-owner-1': {
        'sunday':    {'isOpen': False, 'startTime': '9:00', 'endTime': '18:00'},
        'monday':    {'isOpen': True,  'startTime': '9:00', 'endTime': '18:00'},
        'tuesday':   {'isOpen': True,  'startTime': '9:00', 'endTime': '18:00'},
        'wednesday': {'isOpen': True,  'startTime': '9:00', 'endTime': '18:00'},
        'thursday':  {'isOpen': True,  'startTime': '9:00', 'endTime': '18:00'},
        'friday':    {'isOpen': True,  'startTime': '9:00', 'endTime': '18:00'},
        'saturday':  {'isOpen': True,  'startTime': '9:00', 'endTime': '15:00'},
        'bufferTime': 15,
        'blockedDates': ['2026-10-31'],
    },
    'salon-owner-2': {
        'sunday':    {'isOpen': False, 'startTime': '9:00', 'endTime': '18:00'},
        'monday':    {'isOpen': False, 'startTime': '9:00', 'endTime': '18:00'},
        'tuesday':   {'isOpen': True,  'startTime': '10:00', 'endTime': '19:00'},
        'wednesday': {'isOpen': True,  'startTime': '10:00', 'endTime': '19:00'},
        'thursday':  {'isOpen': True,  'startTime': '10:00', 'endTime': '19:00'},
        'friday':    {'isOpen': True,  'startTime': '10:00', 'endTime': '19:00'},
        'saturday':  {'isOpen': True,  'startTime': '10:00', 'endTime': '16:00'},
        'bufferTime': 30,
        'blockedDates': [],
    },
}

# appointments collection: every document carries its ownerId
MOCK_APPOINTMENTS = [
    {'id': 'a1', 'ownerId': 'salon-owner-1', 'clientName': 'Jordan Lee', 'clientPhone': '555-0101',
     'startTime': '2026-10-17T10:00:00', 'endTime': '2026-10-17T11:00:00', 'status': 'confirmed'},
    {'id': 'a2', 'ownerId': 'salon-owner-1', 'clientName': 'Sam Rivera', 'clientPhone': '555-0102',
     'startTime': '2026-10-17T13:00:00', 'endTime': '2026-10-17T14:30:00', 'status': 'pending'},
    # cancelled, so it must not block any slots
    {'id': 'a3', 'ownerId': 'salon-owner-1', 'clientName': 'Alex Kim', 'clientPhone': '555-0103',
     'startTime': '2026-10-17T12:00:00', 'endTime': '2026-10-17T12:45:00', 'status': 'cancelled'},
    # same day, different salon, so it must never affect salon-owner-1's slots
    {'id': 'a4', 'ownerId': 'salon-owner-2', 'clientName': 'Taylor Brooks', 'clientPhone': '555-0201',
     'startTime': '2026-10-17T11:00:00', 'endTime': '2026-10-17T13:00:00', 'status': 'confirmed'},
]
print('salons:', list(MOCK_AVAILABILITY_BY_OWNER), '| services:', len(SERVICE_DURATIONS),
      '| add-ons:', len(ADD_ON_DURATIONS), '| appointments:', len(MOCK_APPOINTMENTS))

### Tool schemas

Same format as Week 4's `TOOLS`, with two safety fields Week 4 didn't need. Gemini only ever sees each tool's name, description and parameters; the extra fields are read by `dispatch()`:
- `access`: every tool is `'read'`. `dispatch()` refuses anything else while `ALLOW_WRITE_TOOLS` is off, so a tool that books, cancels or blocks a date can't run by accident if one is added later.
- `owner_scoped`: these tools read one salon's data. None of their parameters is an owner id; `dispatch()` adds `OWNER_ID` itself.

Week 4's failure was the model guessing file paths that didn't exist, and the fix was listing the real paths in the tool description. The id lists here apply that fix up front, as `enum`s, so the model picks from the real service ids instead of inventing one like `'gel-mani'`.

In [ ]:
TOOLS = [
    {
        'name': 'search_docs',
        'description': 'Search the NailSalonApp documentation for how a feature works or why it was built '
                       'that way. Returns the most relevant passages and the doc each one came from.',
        'access': 'read',
        'owner_scoped': False,  # the docs describe the app, not one salon
        'parameters': {
            'type': 'object',
            'properties': {'query': {'type': 'string'}},
            'required': ['query'],
        },
    },
    {
        'name': 'get_salon_settings',
        'description': "Look up this owner's current settings: weekly open hours, buffer time between "
                       'appointments, blocked dates, and the service menu with durations in minutes.',
        'access': 'read',
        'owner_scoped': True,
        # no 'parameters' key: Gemini rejects an object schema with no properties
    },
    {
        'name': 'get_available_slots',
        'description': 'List the open appointment start times on one date for a set of services, '
                       'using the same rules as the booking app. date is YYYY-MM-DD.',
        'access': 'read',
        'owner_scoped': True,
        'parameters': {
            'type': 'object',
            'properties': {
                'date': {'type': 'string'},
                'service_ids': {'type': 'array', 'items': {'type': 'string', 'enum': sorted(SERVICE_DURATIONS)}},
                'add_on_ids': {'type': 'array', 'items': {'type': 'string', 'enum': sorted(ADD_ON_DURATIONS)}},
                'nail_length_id': {'type': 'string', 'enum': list(NAIL_LENGTH_EXTRA_MINUTES)},
            },
            'required': ['date', 'service_ids'],
        },
    },
    {
        'name': 'run_python',
        'description': 'Run a single allowlisted arithmetic expression, e.g. to add up durations '
                       'or count hours (guarded code-runner).',
        'access': 'read',
        'owner_scoped': False,
        'parameters': {
            'type': 'object',
            'properties': {'expression': {'type': 'string'}},
            'required': ['expression'],
        },
    },
]
print('tools:', [tool['name'] for tool in TOOLS])

### Time slot tool, ported from NailSalonApp's `timeSlots.ts`

A Python port of `getAvailableSlots`, `calculateTotalDuration` and `formatTimeSlot` from [`src/utils/timeSlots.ts`](https://github.com/BenMillerDev/NailSalonApp/blob/4df756e2504da6ad1782388ca573c330d029238b/src/utils/timeSlots.ts), plus `getBookingsForDate` from [`booking-web/src/services/bookingService.ts`](https://github.com/BenMillerDev/NailSalonApp/blob/4df756e2504da6ad1782388ca573c330d029238b/booking-web/src/services/bookingService.ts), which is how the booking site gathers the appointments it passes in. So the assistant gets the same answer the app would instead of redoing the scheduling math itself. Function names match the TypeScript in snake_case. It differs from the app in four ways:
- The Firestore reads are lookups into the mock data, filtered the same way the app's queries are: by `ownerId`, then date and status.
- "now" is the fixed `NOW` constant instead of the clock, so eval results are reproducible.
- `calculate_total_duration` takes ids, not `Service` objects, because ids are what the model can send.
- The owner's daily booking cap (`maxBookingsPerDay`) isn't applied. `getAvailableSlots` doesn't apply it either.

In [ ]:
def get_availability(owner_id):
    """One salon's availability document (availability/{ownerId} in Firestore)."""
    if owner_id not in MOCK_AVAILABILITY_BY_OWNER:
        raise ValueError('no availability settings found for this salon')
    return MOCK_AVAILABILITY_BY_OWNER[owner_id]


def get_bookings_for_date(owner_id, day):
    """
    Port of getBookingsForDate: this salon's appointments on this day that still
    hold their time. Like the app's query, it returns only id and times, so client
    names and phone numbers never reach the slot math or the model.
    """
    bookings = []
    for appointment in MOCK_APPOINTMENTS:
        is_this_salon = appointment['ownerId'] == owner_id
        is_this_day = datetime.fromisoformat(appointment['startTime']).date() == day.date()
        holds_its_time = appointment['status'] in BOOKED_STATUSES
        if is_this_salon and is_this_day and holds_its_time:
            bookings.append({'id': appointment['id'],
                             'startTime': appointment['startTime'],
                             'endTime': appointment['endTime']})
    return bookings


def get_day_name(day):
    """Return the lowercase weekday name ('monday', ...) used as the availability key."""
    return day.strftime('%A').lower()


def parse_time_string(day, time_string):
    """Combine a date with an 'H:MM' string from the owner's hours into a datetime."""
    hours, minutes = time_string.split(':')
    return datetime(day.year, day.month, day.day, int(hours), int(minutes))


def get_blocked_ranges(day, bookings, buffer_minutes):
    """
    Turn this day's existing bookings into blocked (start, end) ranges, padded
    with the buffer on BOTH sides -- so a new appointment can't start right
    after, or end right before, an existing one.
    """
    buffer = timedelta(minutes=buffer_minutes)
    blocked_ranges = []
    for booking in bookings:
        booking_start = datetime.fromisoformat(booking['startTime'])
        booking_end = datetime.fromisoformat(booking['endTime'])
        if booking_start.date() == day.date():
            blocked_ranges.append((booking_start - buffer, booking_end + buffer))
    return blocked_ranges


def overlaps_with_blocked(slot_start, slot_end, blocked_ranges):
    """
    True if the slot overlaps any blocked range. Comparisons are strict, so a
    slot that ends exactly when a blocked range begins is still allowed.
    """
    for range_start, range_end in blocked_ranges:
        if slot_start < range_end and slot_end > range_start:
            return True
    return False


def get_available_slots(day, total_duration, availability, existing_bookings, now=NOW):
    """
    Every start time, in 15-minute steps, where the full appointment fits inside
    the owner's hours without overlapping a buffered existing booking.
    """
    # Step 1 -- the owner isn't working this day of the week
    day_availability = availability.get(get_day_name(day))
    if day_availability is None or not day_availability['isOpen']:
        return []

    # Step 2 -- the owner manually blocked this specific date
    # (timeSlots.ts checks this after building blocked ranges; same result)
    if day.strftime('%Y-%m-%d') in availability.get('blockedDates', []):
        return []

    # Step 3 -- existing bookings, padded with the owner's buffer
    buffer_minutes = availability.get('bufferTime', DEFAULT_BUFFER_MINUTES)
    blocked_ranges = get_blocked_ranges(day, existing_bookings, buffer_minutes)

    # Step 4 -- walk every candidate start time across the work day
    work_start = parse_time_string(day, day_availability['startTime'])
    work_end = parse_time_string(day, day_availability['endTime'])
    appointment_length = timedelta(minutes=total_duration)
    step = timedelta(minutes=SLOT_INCREMENT_MINUTES)

    slots = []
    current = work_start
    while current < work_end:
        slot_end = current + appointment_length
        fits_before_close = slot_end <= work_end
        if fits_before_close and not overlaps_with_blocked(current, slot_end, blocked_ranges):
            slots.append(current)
        current = current + step

    # Step 5 -- same-day bookings must start more than 1 hour from now
    if day.date() == now.date():
        minimum_start = now + timedelta(minutes=MINIMUM_ADVANCE_MINUTES)
        slots = [slot for slot in slots if slot > minimum_start]

    return slots


def calculate_total_duration(service_ids, add_on_ids=None, nail_length_id=None):
    """Total appointment length in minutes: services + add-ons + any extra time for longer nails."""
    add_on_ids = add_on_ids or []  # add-ons are optional, so the model may leave them out

    services_minutes = 0
    for service_id in service_ids:
        services_minutes += SERVICE_DURATIONS[service_id]

    add_ons_minutes = 0
    for add_on_id in add_on_ids:
        add_ons_minutes += ADD_ON_DURATIONS[add_on_id]

    length_minutes = NAIL_LENGTH_EXTRA_MINUTES.get(nail_length_id, 0)  # no length chosen = no extra time
    return services_minutes + add_ons_minutes + length_minutes


def format_time_slot(slot):
    """datetime -> '2:30 PM', the same format the app shows customers."""
    return slot.strftime('%I:%M %p').lstrip('0')


def get_available_slots_tool(owner_id, date, service_ids, add_on_ids=None, nail_length_id=None):
    """
    What dispatch() calls for get_available_slots. owner_id comes from dispatch(),
    not the model; everything else is the model's plain JSON arguments. Returns
    something JSON-serializable. A malformed date raises ValueError, which
    dispatch() reports back to the model as an execution_error.
    """
    day = datetime.strptime(date, '%Y-%m-%d')
    total_duration = calculate_total_duration(service_ids, add_on_ids, nail_length_id)
    availability = get_availability(owner_id)
    existing_bookings = get_bookings_for_date(owner_id, day)
    slots = get_available_slots(day, total_duration, availability, existing_bookings)
    return {
        'date': date,
        'weekday': get_day_name(day),
        'total_duration_minutes': total_duration,
        'slot_count': len(slots),
        'slots': [format_time_slot(slot) for slot in slots],
    }

### Salon settings and guarded calculator tools

`run_python` is reused from Week 4, spread over more lines with longer names but the same logic. One small fix: an operator outside the allowlist (like `%`) now gets a clear error instead of a bare `KeyError`. `get_salon_settings` plays the role Week 4's `get_file_stats` did: a plain lookup into the mock data, here for one salon only.

In [ ]:
ALLOWED_OPERATORS = {
    ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul,
    ast.Div: operator.truediv, ast.Pow: operator.pow, ast.USub: operator.neg,
}


def get_operator_function(operator_node):
    """The Python function for an allowlisted operator, or a clear error for any other."""
    operator_type = type(operator_node)
    if operator_type not in ALLOWED_OPERATORS:
        raise ValueError(f'operator not allowed: {operator_type.__name__}')
    return ALLOWED_OPERATORS[operator_type]


def evaluate_arithmetic(node):
    """
    Compute a parsed expression one piece at a time, allowing only numbers and
    the operators above. Anything else -- names, function calls, imports,
    attribute access -- raises, which is what makes this code-runner safe.
    """
    is_number = isinstance(node, ast.Constant) and type(node.value) in (int, float)
    if is_number:
        return node.value
    if isinstance(node, ast.BinOp):  # e.g. 60 + 15
        apply_operator = get_operator_function(node.op)
        return apply_operator(evaluate_arithmetic(node.left), evaluate_arithmetic(node.right))
    if isinstance(node, ast.UnaryOp):  # e.g. -15
        apply_operator = get_operator_function(node.op)
        return apply_operator(evaluate_arithmetic(node.operand))
    raise ValueError('only arithmetic is allowed')


class ExpressionTimeout(Exception): pass


def raise_timeout(signal_number, stack_frame):
    """Called by the alarm signal when an expression runs too long (e.g. a huge power like 9**9**9)."""
    raise ExpressionTimeout('expression took too long to evaluate')


def run_python(expression, timeout_seconds=2):
    """The run_python tool: evaluate one arithmetic expression, giving up after timeout_seconds."""
    signal.signal(signal.SIGALRM, raise_timeout)
    signal.alarm(timeout_seconds)
    try:
        parsed_expression = ast.parse(expression, mode='eval')
        return evaluate_arithmetic(parsed_expression.body)
    finally:
        signal.alarm(0)  # cancel so a stale alarm can't fire on some later, unrelated call


def get_salon_settings(owner_id):
    """
    One salon's settings as the model needs them: hours, buffer, blocked dates,
    and service lengths. owner_id comes from dispatch(), not the model.
    """
    availability = get_availability(owner_id)
    weekdays = ['monday', 'tuesday', 'wednesday', 'thursday', 'friday', 'saturday', 'sunday']
    weekly_hours = {}
    for day_name in weekdays:
        weekly_hours[day_name] = availability[day_name]
    return {
        'weekly_hours': weekly_hours,
        'buffer_minutes': availability['bufferTime'],
        'blocked_dates': availability['blockedDates'],
        'service_minutes': SERVICE_DURATIONS,
        'add_on_minutes': ADD_ON_DURATIONS,
        'nail_length_extra_minutes': NAIL_LENGTH_EXTRA_MINUTES,
    }

# tool name -> the Python function dispatch() calls for it
IMPL = {
    'search_docs': search_docs,
    'get_salon_settings': get_salon_settings,
    'get_available_slots': get_available_slots_tool,
    'run_python': run_python,
}

### Argument validation

Reused from Week 4 and split into one small check per kind of argument, so each rule reads on its own. The one new rule is `check_list_of_ids`, because `service_ids` and `add_on_ids` are lists and Week 4's tools only took strings and numbers. Every error message goes back to the model, so it says exactly what to fix.

In [ ]:
class ToolArgError(Exception): pass       # the model sent bad arguments; it can fix them and retry
class ToolNotPermitted(Exception): pass   # the tool isn't allowed to run at all


def find_tool(tool_name):
    """The TOOLS entry for this name. An unknown name is the model's mistake, so it's a ToolArgError."""
    for tool in TOOLS:
        if tool['name'] == tool_name:
            return tool
    raise ToolArgError(f'unknown tool: {tool_name}')


def check_string(field_name, value):
    """Raise unless the value is a string."""
    if not isinstance(value, str):
        raise ToolArgError(f'{field_name} must be a string')


def check_number(field_name, value):
    """Raise unless the value is a real, finite number."""
    # type() rather than isinstance(), because Python counts True and False as ints
    if type(value) not in (int, float):
        raise ToolArgError(f'{field_name} must be a number (not a boolean)')
    if isinstance(value, float) and not math.isfinite(value):
        raise ToolArgError(f'{field_name} must be finite')


def check_list_of_ids(field_name, value, item_spec):
    """
    New in Week 8: raise unless the value is a list of string ids from the
    schema's allowed list. Lists of ids are the only arrays TOOLS uses.
    """
    if item_spec['type'] != 'string':
        raise ValueError(f'extend validate() to support arrays of: {item_spec["type"]}')
    if not isinstance(value, list):
        raise ToolArgError(f'{field_name} must be a list')

    allowed_ids = item_spec.get('enum')
    for item in value:
        if not isinstance(item, str):
            raise ToolArgError(f'every item in {field_name} must be a string')
        if allowed_ids is not None and item not in allowed_ids:
            raise ToolArgError(f'{field_name} item {item!r} not in {allowed_ids}')


def check_field(field_name, value, field_spec):
    """Check one argument against its schema: the type first, then the allowed values if the schema lists them."""
    field_type = field_spec['type']
    if field_type == 'string':
        check_string(field_name, value)
    elif field_type == 'number':
        check_number(field_name, value)
    elif field_type == 'array':
        check_list_of_ids(field_name, value, field_spec['items'])
    else:
        raise ValueError(f'extend validate() to support schema type: {field_type}')

    allowed_values = field_spec.get('enum')
    if allowed_values is not None and value not in allowed_values:
        raise ToolArgError(f'{field_name}={value!r} not in {allowed_values}')


def validate(tool, arguments):
    """Raise ToolArgError if the model's arguments don't match this tool's schema."""
    no_arguments_schema = {'type': 'object', 'properties': {}}
    schema = tool.get('parameters', no_arguments_schema)  # get_salon_settings has no parameters
    if not isinstance(arguments, dict):
        raise ToolArgError('arguments must be a JSON object')

    for required_field in schema.get('required', []):
        if required_field not in arguments:
            raise ToolArgError(f'missing required field: {required_field}')

    for field_name, value in arguments.items():
        field_spec = schema['properties'].get(field_name)
        if field_spec is None:
            # this is also what stops a model-supplied owner_id: no schema declares one
            raise ToolArgError(f'unexpected field: {field_name}')
        check_field(field_name, value, field_spec)

### Dispatch with the safety rules

Week 4's `dispatch()`, with two safety rules added:
- **Read-only.** It refuses any tool not marked `'access': 'read'`. A tool with no `access` field counts as one that changes data, so a new tool is blocked until someone marks it read-only on purpose.
- **Owner-scoped.** It takes the signed-in owner's id from its caller and passes it to owner-scoped tools itself. The model can't send one: `validate()` rejects an `owner_id` argument as an unexpected field.

In [ ]:
def check_permitted(tool):
    """Read-only for now: raise unless the tool is explicitly marked 'access': 'read'."""
    is_read_only = tool.get('access') == 'read'
    if not is_read_only and not ALLOW_WRITE_TOOLS:
        raise ToolNotPermitted(f"{tool['name']} can change data, and this assistant is read-only")


def run_tool(tool, arguments, owner_id):
    """Call the tool's function, adding owner_id for tools that read one salon's data."""
    tool_function = IMPL[tool['name']]
    if tool['owner_scoped']:
        return tool_function(owner_id=owner_id, **arguments)
    return tool_function(**arguments)


def dispatch(tool_name, arguments, owner_id):
    """
    Run one tool call from the model and return a result it can read, success or
    failure. owner_id has no default on purpose: every call has to say which
    signed-in owner it's acting for.
    """
    try:
        tool = find_tool(tool_name)
        check_permitted(tool)
        validate(tool, arguments)
        output = run_tool(tool, arguments, owner_id)
        json.dumps(output, allow_nan=False)  # reject results the model cannot receive as JSON
        return {'ok': True, 'tool': tool_name, 'output': output}
    except ToolArgError as error:
        return {'ok': False, 'tool': tool_name, 'error_type': 'invalid_arguments', 'message': str(error)}
    except ToolNotPermitted as error:
        return {'ok': False, 'tool': tool_name, 'error_type': 'not_permitted', 'message': str(error)}
    except Exception as error:
        return {'ok': False, 'tool': tool_name, 'error_type': 'execution_error', 'message': str(error)}

### Sanity-check the tools without the model

Scripted calls, like Week 4's, so every tool and error path is checked before the model is involved. The expected slots for Saturday 10/17 were worked out by hand: with the 15-minute buffer, the 10:00 and 1:00 bookings block 9:45–11:15 and 12:45–2:45, and the salon closes at 3:00, so a 60-minute gel manicure can only start at 11:15, 11:30 or 11:45. Adding nail art and long nails makes it 90 minutes, which fills the 11:15–12:45 gap exactly, so only 11:15 is left. That case checks that a slot ending right where a blocked range starts is still allowed, as in `timeSlots.ts`. Salon 1's cancelled 12:00 appointment and salon 2's 11:00–1:00 appointment are on the same day, and either one would wipe out those slots if the tool counted it.

In [ ]:
scripted = [
  ('search_docs',         {'query':'How far in advance must a customer book?'}),                    # expect time_slot_algorithm
  ('get_available_slots', {'date':'2026-10-17','service_ids':['gel-manicure']}),                    # expect 11:15, 11:30, 11:45 AM
  ('get_available_slots', {'date':'2026-10-17','service_ids':['gel-manicure'],
                           'add_on_ids':['nail-art'],'nail_length_id':'long'}),                     # 90 min fills 11:15-12:45 exactly: expect 11:15 AM only
  ('get_available_slots', {'date':'2026-10-18','service_ids':['gel-manicure']}),                    # Sunday, closed
  ('get_available_slots', {'date':'2026-10-31','service_ids':['gel-manicure']}),                    # blocked date
  ('get_available_slots', {'date':'2026-10-12','service_ids':['classic-manicure']}),                # today: nothing before 11:15
  ('get_available_slots', {'date':'2026-10-17','service_ids':['gel-mani']}),                        # scripted invalid id
  ('get_available_slots', {'date':'2026-10-17','service_ids':'gel-manicure'}),                      # scripted string, not a list
  ('get_available_slots', {'date':'10/17/2026','service_ids':['gel-manicure']}),                    # scripted wrong date format
  ('get_salon_settings',  {}),
  ('run_python',          {'expression':'60+15+15'}),
  ('run_python',          {'expression':'__import__("os").system("echo hi")'}),                     # guarded
]
for tool_name, arguments in scripted:
    result = dispatch(tool_name, arguments, owner_id=OWNER_ID)
    status = 'OK ' if result['ok'] else 'ERR'
    print(f'[{status}] {tool_name}({arguments}) -> {json.dumps(result, allow_nan=False)}')

### Check the safety rules

The same Saturday question asked as `salon-owner-2` should get that salon's answer. It's open 10:00–4:00 with a 30-minute buffer, so its 11:00–1:00 appointment blocks 10:30–1:30, and a 60-minute gel manicure can start from 1:30 to 3:00. The other two calls are the attempts the rules exist to stop: the model naming another salon in its arguments, and a request for a salon that doesn't exist.

In [ ]:
safety_checks = [
  ('get_available_slots', {'date':'2026-10-17','service_ids':['gel-manicure']}, 'salon-owner-2'),   # expect 1:30 PM to 3:00 PM
  ('get_salon_settings',  {'owner_id':'salon-owner-2'},                         OWNER_ID),          # model tries to pick the salon
  ('get_salon_settings',  {},                                                   'salon-owner-9'),   # no such salon
]
for tool_name, arguments, owner_id in safety_checks:
    result = dispatch(tool_name, arguments, owner_id=owner_id)
    status = 'OK ' if result['ok'] else 'ERR'
    print(f'[{status}] as {owner_id}: {tool_name}({arguments}) -> {json.dumps(result, allow_nan=False)}')

## Agent: tools + system prompt (Weeks 3 and 4)

### Load the versioned system prompt

Week 3's pattern: the prompt lives in a versioned file (`prompts/assistant_v1.md`), and the output is a JSON object with a rationale, so a prompt change shows up as a reviewable diff. Like the Week 3 prompts, it has few-shot examples. Their questions (a pedicure on the 20th, deposits, exporting) are kept out of Part 2's held-out set so the evaluation doesn't test the examples.

In [ ]:
def load_prompt(version):
    # prompt files are the actual versioned artifact -- read from disk the
    # same way any other file under version control would be
    prompt_path = PROMPTS_DIR / f'assistant_{version}.md'
    return prompt_path.read_text()


def build_system_prompt(prompt_file_text, now):
    """
    Keep everything from the '## System' heading down (the notes above it are
    for people reading the repo, not the model), and fill in today's date so
    the model can turn 'this Saturday' into a real date.
    """
    _, system_prompt = prompt_file_text.split('## System', 1)
    today_text = now.strftime('%A, %B %d, %Y, %I:%M %p')  # e.g. Monday, October 12, 2026, 10:00 AM
    return system_prompt.strip().replace('{today}', today_text)


SYSTEM_PROMPT = build_system_prompt(load_prompt(PROMPT_VERSION), NOW)
print(f'{PROMPT_VERSION} system prompt: {len(SYSTEM_PROMPT)} chars')
print()
print('--- first 400 chars ---')
print(SYSTEM_PROMPT[:400])

### Connect the tools to Gemini

Reused from Week 4. Gemini only sees each tool's name, description and parameters; the `access` and `owner_scoped` fields stay on this side for `dispatch()`.

In [ ]:
!pip install -q google-genai
from google import genai
from google.genai import types

gemini_client = genai.Client(api_key=os.environ['GEMINI_API_KEY']) if HAS_API_KEY else None


def build_function_declaration(tool):
    """One TOOLS entry as Gemini sees it."""
    return types.FunctionDeclaration(
        name=tool['name'],
        description=tool['description'],
        parameters=tool.get('parameters'),  # None for get_salon_settings, which takes no arguments
    )


function_declarations = [build_function_declaration(tool) for tool in TOOLS]
gemini_tool = types.Tool(function_declarations=function_declarations)
print('gemini tool built from', len(gemini_tool.function_declarations), 'schemas; client ready:', gemini_client is not None)

### Agent loop

Week 4's `run_agent`, with three changes: it sends the system prompt and `TEMPERATURE` with every turn, it takes the signed-in owner's id and hands it to `dispatch()`, and its default turn limit is the `MAX_TURNS` constant.

In [ ]:
def _ask_model(contents):
    # Send the conversation so far to Gemini and return its response for
    # this turn. Tool results already appended to `contents` are just more
    # turns in the same history -- Gemini doesn't need them called out
    # specially.
    return gemini_client.models.generate_content(
        model=MODEL,
        contents=contents,
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_PROMPT,
            temperature=TEMPERATURE,
            tools=[gemini_tool],
            automatic_function_calling=types.AutomaticFunctionCallingConfig(disable=True),
        ),
    )


def _requested_calls(candidate):
    # The tool calls Gemini proposed this turn, if any -- empty once it's
    # ready to give a final answer instead.
    if candidate.content is None or not candidate.content.parts:
        return []
    return [part.function_call for part in candidate.content.parts if part.function_call]


def _run_requested_call(call, turn, owner_id, log, verbose):
    # Execute one proposed call through dispatch(), record it in `log`,
    # and return it as a FunctionResponse Part carrying the same call id Gemini
    # sent -- that id is what lets Gemini match this result back to its call.
    args = call.args or {}
    result = dispatch(call.name, args, owner_id)
    log.append({'turn': turn, 'call_id': call.id, 'name': call.name, 'args': args, 'result': result})
    if verbose:
        status = 'OK ' if result['ok'] else 'ERR'
        print(f'[turn {turn}] [{status}] {call.name}({args}) -> {result}')
    return types.Part(function_response=types.FunctionResponse(id=call.id, name=call.name, response=result))


def run_agent(query, owner_id, max_turns=MAX_TURNS, verbose=True):
    # Send `query` to Gemini with TOOLS attached, intercept every tool
    # call it makes, run each one through dispatch() as `owner_id`, and feed
    # the results back -- tagged with the model's own call ids -- until it
    # gives a final text answer (or max_turns is hit).
    contents = [types.Content(role='user', parts=[types.Part(text=query)])]
    log = []

    for turn in range(max_turns):
        response = _ask_model(contents)
        candidate = response.candidates[0]
        contents.append(candidate.content)  # the model's own turn, calls included

        calls = _requested_calls(candidate)
        if not calls:
            final_text = response.text or ''
            log.append({'turn': turn, 'final_text': final_text})
            if verbose: print(f'[turn {turn}] final answer: {final_text}')
            return final_text, log

        tool_response_parts = [_run_requested_call(call, turn, owner_id, log, verbose) for call in calls]
        contents.append(types.Content(role='user', parts=tool_response_parts))

    final_text = f'[gave up: no final answer after {max_turns} turns]'
    log.append({'turn': max_turns, 'gave_up': True, 'final_text': final_text})
    if verbose: print(f'[gave up] no final answer after {max_turns} turns')
    return final_text, log

### Parse the JSON answer

`extract_json_object` is from Week 3. If the reply still isn't valid JSON, the raw text is kept as the answer and flagged, so a formatting slip shows up in Part 2's metrics instead of crashing the run.

In [ ]:
def extract_json_object(text):
    # models sometimes wrap the JSON in a ```json fence, or add prose before or
    # after it, even when told not to -- pull out just the {...} object so
    # json.loads can parse it regardless of what's around it
    fenced_match = re.search(r'```(?:json)?\s*(\{.*?\})\s*```', text, re.DOTALL)
    if fenced_match:
        return fenced_match.group(1)
    braces_match = re.search(r'\{.*\}', text, re.DOTALL)
    if braces_match:
        return braces_match.group(0)
    return text


def parse_answer(final_text):
    """The model's final reply as answer, sources and rationale, plus a parse_error flag."""
    try:
        parsed_reply = json.loads(extract_json_object(final_text))
    except json.JSONDecodeError:
        parsed_reply = None

    if not isinstance(parsed_reply, dict):
        return {'answer': final_text, 'sources': [], 'rationale': '', 'parse_error': True}

    return {
        'answer': parsed_reply.get('answer', ''),
        'sources': parsed_reply.get('sources', []),
        'rationale': parsed_reply.get('rationale', ''),
        'parse_error': False,
    }

### Ask the assistant, reusing saved runs

`ask_assistant` is the one entry point the rest of the notebook uses. A run is saved under a key made of every setting that can change it, so flipping `USE_HYDE` for an ablation gets a fresh run instead of a stale saved one. `show_run` prints a run's tool calls in order, then its answer; Part 3 reuses it to trace a failure.

In [ ]:
def run_cache_key(question, owner_id):
    """Every setting that can change a run's result, joined into one string."""
    settings = [MODEL, PROMPT_VERSION, f'hyde={USE_HYDE}', f'rerank={USE_RERANK}', owner_id, question]
    return '|'.join(settings)


def ask_assistant(question, owner_id):
    """
    One full assistant run: the saved one if it exists, otherwise a live run that
    gets saved. Returns None when there's no API key and nothing saved.
    """
    run_cache = load_cache('runs')
    cache_key = run_cache_key(question, owner_id)
    if cache_key in run_cache:
        return run_cache[cache_key]
    if not HAS_API_KEY:
        return None

    final_text, call_log = run_agent(question, owner_id, verbose=False)
    parsed_answer = parse_answer(final_text)
    run = {
        'question': question,
        'owner_id': owner_id,
        'answer': parsed_answer['answer'],
        'sources': parsed_answer['sources'],
        'rationale': parsed_answer['rationale'],
        'parse_error': parsed_answer['parse_error'],
        'final_text': final_text,
        'call_log': call_log,
    }
    # round-trip through JSON (default=str turns any unusual SDK type into text) so a
    # live run looks exactly like one loaded from the cache later
    run = json.loads(json.dumps(run, default=str))
    run_cache[cache_key] = run
    save_cache('runs', run_cache)
    return run


def summarize_tool_result(result):
    """A one-line view of a tool result, short enough to scan in a log."""
    if not result['ok']:
        return f"{result['error_type']}: {result['message']}"
    output = result['output']
    if result['tool'] == 'search_docs':
        retrieved_docs = [passage['doc'] for passage in output['passages']]
        return f'passages from {retrieved_docs}'
    if result['tool'] == 'get_available_slots':
        return f"{output['slot_count']} slots on {output['date']} ({output['total_duration_minutes']} min): {output['slots']}"
    if result['tool'] == 'get_salon_settings':
        return 'hours, buffer, blocked dates, and service lengths'
    return str(output)


def show_run(run):
    """Print one run: the question, every tool call in order, then the answer, sources and rationale."""
    print('QUESTION:', run['question'])
    for log_entry in run['call_log']:
        if 'name' not in log_entry:
            continue  # the final-answer entry; the parsed answer is printed below
        status = 'OK ' if log_entry['result']['ok'] else 'ERR'
        print(f"  [turn {log_entry['turn']}] [{status}] {log_entry['name']}({log_entry['args']})")
        print(f"      -> {summarize_tool_result(log_entry['result'])}")
    print('ANSWER:', textwrap.fill(run['answer'], width=100, subsequent_indent='        '))
    print('SOURCES:', run['sources'])
    print('RATIONALE:', run['rationale'])
    if run['parse_error']:
        print('(reply was not valid JSON; the raw text is shown as the answer)')

### Example questions

Four questions that each lean on a different part of the system. The first needs two techniques in one answer: the slot tool for the open times and the docs for why the time after a booking is blocked. These are demonstrations only; Part 2's held-out questions are written separately.

| Question | Should use | Expected answer |
|---|---|---|
| Openings this Saturday for a gel manicure with nail art, and why not right after 10:00 | `get_available_slots`, `search_docs` | 11:15 and 11:30 AM (75 minutes fits the 11:15–12:45 gap twice); the buffer between appointments |
| Saturday hours and blocked dates this month | `get_salon_settings` | 9:00 AM–3:00 PM; October 31 is blocked |
| Text message confirmations | `search_docs` | Planned via Twilio, not available yet |
| Block off November 2 | none | Can't make changes; read-only |

In [ ]:
EXAMPLE_QUESTIONS = [
    "Do I have any openings this Saturday for a gel manicure with nail art? And why can't anyone book right after my 10 o'clock?",
    "What are my hours on Saturdays, and do I have any days blocked off this month?",
    "Can my clients get a text message confirmation when they book?",
    "Please block off November 2nd for me, I'll be out of town.",
]

example_runs = []
for question in EXAMPLE_QUESTIONS:
    run = ask_assistant(question, owner_id=OWNER_ID)
    if run is None:
        print('[API-BLOCKED] no saved run for:', question)
        continue
    example_runs.append(run)
    show_run(run)
    print('-' * 100)